# 00 — Kunz `.mat` file audit

This notebook inspects file structure **without guessing field meanings or loading full neural arrays**. Run it after placing or symlinking one representative Kunz file in `data/raw/`.

The output is the evidence needed to implement the real adapter: storage version, variable paths, shapes, dtypes, labels, timing fields, and axis order.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'pyproject.toml').exists():
    raise RuntimeError('Run this notebook from the repository root or notebooks directory.')
sys.path.insert(0, str(ROOT / 'src'))

from kunz_speech_geometry.config import load_config
from kunz_speech_geometry.io import audit_directory, inspect_mat_file, sha256_file

config = load_config(ROOT / 'configs' / 'default.yaml')
raw_dir = ROOT / config['dataset']['raw_dir']
raw_dir

PosixPath('/Users/ronnuriel/Documents/Codex/2026-08-20/referenced-chatgpt-conversation-this-is-an/outputs/kunz-neural-speech-geometry/data/raw')

In [2]:
mat_files = sorted(raw_dir.rglob('*.mat'))
print(f'Found {len(mat_files)} .mat file(s) under {raw_dir}')
for path in mat_files[:20]:
    print(' -', path.relative_to(ROOT))
if len(mat_files) > 20:
    print(f' ... and {len(mat_files) - 20} more')

Found 0 .mat file(s) under /Users/ronnuriel/Documents/Codex/2026-08-20/referenced-chatgpt-conversation-this-is-an/outputs/kunz-neural-speech-geometry/data/raw


In [3]:
# Header/HDF5 metadata only; no full neural matrix is loaded.
audit = audit_directory(raw_dir, include_checksum=False)
audit

,file,name,shape_on_disk,class_or_dtype,storage


## Adapter checklist

Before writing any loader, answer these from the source README and audit output:

- Which field is threshold crossings, and which is spike-band power?
- Are values counts/bin, rate, power, or already normalized?
- What is the exact axis order? Do not infer it only from familiar channel counts.
- What event defines 0 ms for attempted speech and passive listening?
- Where are participant, session, block, trial, behavior, and word labels?
- Are bad channels or rejected trials supplied?
- Are attempted and listening both present within each session/block?
- Does T16 use attempted mimed rather than attempted vocalized speech in this file?

Compute SHA-256 only when preparing a manifest; multi-gigabyte files can take time. Example: `sha256_file(mat_files[0])`.